<Note> Before reading this section, ensure you are familiar with the [Linear Algebra Annex](/aiml-common/lectures/ml-math/linear-algebra). After reading this section, consider doing [this assignment.](https://colab.research.google.com/github/pantelis/eng-ai-agents/blob/main/notebooks/assignments/dim-reduction/low_rank_gaussians_1.ipynb) </Note>

<CardGroup cols={2}>
  <Card title="Assignment" icon="graduation-cap" href="https://colab.research.google.com/github/pantelis/eng-ai-agents/blob/main/notebooks/assignments/dim-reduction/low_rank_gaussians_1.ipynb">
    Explore low-rank Gaussian geometry with guided exercises
  </Card>
  {/* <Card title="Graduate Assignment" icon="flask" href="https://colab.research.google.com/github/pantelis/eng-ai-agents/blob/main/notebooks/assignments/dim-reduction/low_rank_gaussians_2.ipynb">
    Advanced analysis of rank-constrained covariance modifications
  </Card> */}
</CardGroup>

Large linear operators appear throughout modern machine learning systems.
Yet empirical evidence shows that meaningful changes to such systems often
occupy only a small number of directions.

This tutorial develops that idea from first principles, using Gaussian
distributions to isolate the geometry of low-rank change before connecting
it to modern architectures like LoRA (Low-Rank Adaptation).

---

## Part I: Covariance and Geometry

### Covariance as shape

Let $x \in \mathbb{R}^d$ be a zero-mean random vector with covariance

$$
\Sigma = \mathbb{E}[x x^\top].
$$

Eigenvectors of $\Sigma$ define orthogonal directions in space, and eigenvalues
measure the variance along those directions.

Geometrically, $\Sigma$ describes the shape of a probability ellipsoid.

<Tip>
Internalize that eigenvalues are not abstract quantities - they directly correspond to observable spread in data.
</Tip>

### Latent variable construction

Instead of specifying $\Sigma$ directly, we construct it from a lower-dimensional
latent variable:

$$
z \sim \mathcal{N}(0, I_k), \quad x = W z + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2 I_d).
$$

This implies

$$
\Sigma = W W^\top + \sigma^2 I_d.
$$

The matrix $W$ determines the *signal subspace*; the noise term fills the
remaining directions isotropically.

<Note>
This is the same construction used in classical factor analysis and probabilistic PCA.
</Note>

### Rank as a geometric constraint

Because $\mathrm{rank}(W W^\top) \le k$, only $k$ directions can carry variance
above the noise level.

This explains why many high-dimensional datasets exhibit:
- Rapidly decaying eigenvalue spectra
- Effective low dimensionality

---

## Part II: Structured Coefficient Changes

### A reference distribution

Fix a reference matrix $W_0$ with covariance

$$
\Sigma_0 = W_0 W_0^\top + \sigma^2 I_d.
$$

This defines a baseline Gaussian distribution.
All subsequent changes are measured relative to this geometry.

### Low-rank coefficient modifications

We now restrict changes to the form

$$
\Delta W = B A,
$$

where:
- $B$ is a $d \times r$ matrix
- $A$ is an $r \times k$ matrix
- $r \ll \min(d,k)$

Only $r$ new directions can be introduced.

<Warning>
Low-rank structure does not limit *how much* the matrix changes, but *where* it can change.
</Warning>

### Diffuse coefficient changes

For comparison, consider dense changes with no preferred directions,
scaled to match the Frobenius norm of the low-rank case.

This contrast isolates the role of structure from magnitude.

### Observable consequences

Empirically, one finds:
- Low-rank changes alter a small number of eigenvalues dramatically
- Diffuse changes alter many eigenvalues modestly

Part III proves the principle behind these observations:

> **A rank-$r$ change to $W$ can change the variance only inside a subspace of dimension at most $2r$.**

---

## Part III: Likelihood and Statistical Geometry

### Empirical covariance

Given samples $\{x_i\}_{i=1}^n$, the empirical covariance

$$
S = \frac{1}{n} \sum_{i=1}^n x_i x_i^\top
$$

is a sufficient statistic for zero-mean Gaussian models.

### Gaussian likelihood geometry

The Gaussian negative log-likelihood is

$$
\mathcal{L}(\Sigma) = \frac{1}{2} \left[ \log \det \Sigma + \mathrm{tr}\!\left(\Sigma^{-1} S\right) \right].
$$

Geometric interpretation:
- $\log \det \Sigma$ penalizes volume mismatch
- $\mathrm{tr}(\Sigma^{-1} S)$ penalizes directional mismatch

<Note>
You may recognize this as a Riemannian geometry on the cone of positive definite matrices.
</Note>

### Rank-constrained covariance matching

Expanding

$$
\Sigma - \Sigma_0 = W_0 \Delta W^\top + \Delta W W_0^\top + \Delta W \Delta W^\top
$$

implies

$$
\mathrm{rank}(\Sigma - \Sigma_0) \le 2r.
$$

Let $U$ be the column space of $\Sigma - \Sigma_0$. It has dimension at most $2r$.
For every direction $v$ orthogonal to $U$, $(\Sigma - \Sigma_0)\, v = 0$, so
$v^\top \Sigma v = v^\top \Sigma_0 v$. The variance along $v$ does not change, whatever the
dimension $d$.

The bound restricts where the variance changes. It does not say that most eigenvectors of
$\Sigma_0$ survive. Take $W_0 = \mathrm{diag}(1, \dots, d)$ and the rank-one change
$\Delta W = \mathbf{1}\mathbf{1}^\top$. Entry $(i, j)$ of $\Sigma - \Sigma_0$ is $i + j + d$, so every
off-diagonal entry of $\Sigma$ is positive and no coordinate axis is still an eigenvector, even
though $\mathrm{rank}(\Sigma - \Sigma_0) \le 2$.

If the target covariance differs from $\Sigma_0$ by a matrix of rank above $2r$, no rank-$r$
change reaches it exactly. This limit explains the likelihood saturation observed in experiments.

---

## Part IV: Connection to LoRA in Transformers

LoRA (Low-Rank Adaptation) applies the same structural assumption to large linear operators:

- A reference matrix is fixed (the pretrained weights)
- Adaptation is constrained to a low-rank subspace
- Rank controls expressive capacity

Transformers obscure this geometry with nonlinearities and attention mechanisms.
The Gaussian setting exposes it directly.

In [ ]:
import numpy as np

# LoRA structure in practice
# Original: y = Wx
# LoRA:     y = Wx + BAx  where B ∈ R^{d×r}, A ∈ R^{r×k}

class LoRALayer:
    def __init__(self, W, rank, rng):
        self.W = W  # frozen pretrained weights
        self.B = np.zeros((W.shape[0], rank))
        self.A = rng.normal(scale=1 / np.sqrt(W.shape[1]), size=(rank, W.shape[1]))

    def forward(self, x):
        return self.W @ x + self.B @ (self.A @ x)

Because $B$ starts at zero, the layer reproduces the frozen map $Wx$ exactly at initialization. Training then updates only $B$ and $A$, which hold $r(d+k)$ numbers instead of the $dk$ numbers of a full update to $W$.

In [ ]:
rng = np.random.default_rng(0)
d, k, r = 1024, 1024, 8

W = rng.normal(size=(d, k))
layer = LoRALayer(W, rank=r, rng=rng)
x = rng.normal(size=k)

print("forward pass equals W @ x at initialization:", np.allclose(layer.forward(x), W @ x))

full = d * k
lora = layer.B.size + layer.A.size
print(f"full update of W: {full:,} parameters")
print(f"LoRA update BA:   {lora:,} parameters ({lora / full:.2%} of full)")

---

## Final Takeaway

Low-rank adaptation is not an optimization trick.
It is a geometric assumption about how complex systems change.

- When that assumption holds, low-rank methods are statistically efficient
- When it does not, no algorithm can avoid higher-dimensional modification

## References

- Hu, E.J., et al. (2021). [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)
- Bishop, C.M. (2006). [Pattern Recognition and Machine Learning](https://www.microsoft.com/en-us/research/uploads/prod/2006/01/Bishop-Pattern-Recognition-and-Machine-Learning-2006.pdf), Chapter 12.2 (Probabilistic PCA)